# GNN Baseline (Multi-PNA-EU) — Colab 실행 노트북

**실행 전 체크리스트**
- [ ] 런타임 → GPU (T4 이상) 설정
- [ ] Google Drive에 `Graph_AML/data/processed/gnn_inputs/` 아래 04번 산출물 3개 존재 확인
  - `formatted_transactions_gf.csv`
  - `formatted_transactions.csv`
  - `account_mapping.csv`
- [ ] **2번 셀** 경로 설정 확인

## 0. GPU 확인

In [18]:
import torch

print(f"PyTorch : {torch.__version__}")
print(f"CUDA    : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU     : {torch.cuda.get_device_name(0)}")
else:
    print("⚠️  GPU가 없습니다. 런타임 → 런타임 유형 변경 → T4 GPU로 전환하세요.")

PyTorch : 2.10.0+cu128
CUDA    : True
GPU     : NVIDIA A100-SXM4-80GB


## 1. Google Drive 마운트

In [19]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 2. 경로 설정 ← 여기만 수정

In [20]:
from pathlib import Path

# ============================================================
# 팀원 Drive 구조에 맞게 수정하세요
# ============================================================
DRIVE_BASE = Path("/content/drive/MyDrive/Graph_AML")
REPO_URL   = "https://github.com/JKYUPSYCHE/Graph_AML"  # 실제 GitHub URL로 변경
BRANCH     = "gnn/baseline-v2"
# ============================================================

GNN_INPUTS = DRIVE_BASE / "data" / "processed" / "gnn_inputs"
MODEL_SAVE = DRIVE_BASE / "experiments" / "multi_pna_eu" / "models"

# 필수 파일 존재 확인
required = [
    GNN_INPUTS / "formatted_transactions_gf.csv",
    GNN_INPUTS / "formatted_transactions.csv",
    GNN_INPUTS / "account_mapping.csv",
]
all_ok = True
for f in required:
    exists = f.exists()
    print(f"{'[OK]' if exists else '[MISSING]'} {f.name}")
    if not exists:
        all_ok = False

if not all_ok:
    raise FileNotFoundError("04_gnn_graph_process.ipynb 산출물이 없습니다. Drive 경로를 확인하세요.")

MODEL_SAVE.mkdir(parents=True, exist_ok=True)
print("\n경로 설정 완료")
print("GNN_INPUTS :", GNN_INPUTS)
print("MODEL_SAVE :", MODEL_SAVE)

[OK] formatted_transactions_gf.csv
[OK] formatted_transactions.csv
[OK] account_mapping.csv

경로 설정 완료
GNN_INPUTS : /content/drive/MyDrive/Graph_AML/data/processed/gnn_inputs
MODEL_SAVE : /content/drive/MyDrive/Graph_AML/experiments/multi_pna_eu/models


## 3. 레포 클론 & PyG 설치

In [21]:
import os

REPO_DIR = Path("/content/Graph_AML")

if not REPO_DIR.exists():
    os.system(f"git clone {REPO_URL} {REPO_DIR}")
    os.system(f"git -C {REPO_DIR} checkout {BRANCH}")
else:
    os.system(f"git -C {REPO_DIR} fetch origin")
    os.system(f"git -C {REPO_DIR} checkout {BRANCH}")
    os.system(f"git -C {REPO_DIR} pull origin {BRANCH}")

print("레포 준비 완료:", REPO_DIR)

레포 준비 완료: /content/Graph_AML


In [22]:
import torch

torch_ver = torch.__version__.split('+')[0]
cuda_tag  = 'cu' + torch.version.cuda.replace('.', '') if torch.cuda.is_available() else 'cpu'
print(f"설치 대상: torch={torch_ver}, cuda={cuda_tag}")

os.system("pip install -q torch_geometric")
os.system(f"pip install -q torch-scatter torch-sparse -f https://data.pyg.org/whl/torch-{torch_ver}+{cuda_tag}.html")
os.system("pip install -q psutil tqdm tensorboard")
print("패키지 설치 완료")

설치 대상: torch=2.10.0, cuda=cu128
패키지 설치 완료


## 4. 모듈 경로 설정

In [23]:
import sys

BASELINE_DIR = REPO_DIR / "gnn" / "baseline"

# baseline 모듈 import 및 model_settings.json 참조를 위해 CWD 변경
os.chdir(BASELINE_DIR)

for p in [str(BASELINE_DIR), str(REPO_DIR)]:
    if p not in sys.path:
        sys.path.insert(0, p)

print("CWD        :", os.getcwd())
print("sys.path[0]:", sys.path[0])

CWD        : /content/Graph_AML/gnn/baseline
sys.path[0]: /content/Graph_AML


## 5. 실험 설정

In [24]:
from types import SimpleNamespace

# ============================================================
# 실험 설정 — 필요에 따라 수정
# ============================================================
args = SimpleNamespace(
    # 모델
    model      = 'pna',          # gin | gat | pna | rgcn
    data       = 'Small_HI',     # TensorBoard run name 등에 사용
    seed       = 42,

    # 학습
    n_epochs   = 1,              #epoch 수
    batch_size = 8192,
    num_neighs = [100, 100],     # 2-hop 이웃 샘플링 수
    patience   = 10,             # early stopping (None이면 비활성화)

    # Multi-PNA-EU 핵심 플래그 (모두 True여야 Multi-PNA-EU)
    emlps      = True,           # Edge Update (EU)
    reverse_mp = True,           # 양방향 MP (Multi)
    ports      = True,           # Port numbering
    tds        = False,          # Time-delta features (False면 비활성화)
    ego        = True,           # 배치 내 seed 엣지의 노드에 ego ID(1) 부여 — 노드 피처에 1차원

    # 저장/추론
    save_model  = True,
    unique_name = 'small_hi_v1',
    finetune    = False,
    inference   = False,

    tqdm = True,
)
# ============================================================

data_config = {
    "paths": {
        "aml_data"      : str(DRIVE_BASE / "data"),
        "gnn_inputs"    : str(GNN_INPUTS),
        "model_to_load" : str(MODEL_SAVE),
        "model_to_save" : str(MODEL_SAVE),
    }
}

print(f"모델    : {args.model}")
print(f"데이터  : {args.data}")
print(f"에폭    : {args.n_epochs}  |  배치: {args.batch_size}  |  patience: {args.patience}")
print(f"플래그  : emlps={args.emlps}, reverse_mp={args.reverse_mp}, ports={args.ports}, tds={args.tds}, ego={args.ego}")

모델    : pna
데이터  : Small_HI
에폭    : 1  |  배치: 8192  |  patience: 10
플래그  : emlps=True, reverse_mp=True, ports=True, tds=False, ego=True


## 6. 데이터 로딩

In [25]:
import time
import logging
from util import set_seed, logger_setup
from data_loading import get_data

logger_setup()
set_seed(args.seed)

logging.info("데이터 로딩 시작")
t1 = time.perf_counter()

tr_data, val_data, te_data, tr_inds, val_inds, te_inds = get_data(args, data_config)

t2 = time.perf_counter()
logging.info(f"데이터 로딩 완료: {t2 - t1:.2f}s")

## 7. 학습

In [26]:
from training import train_gnn

logging.info("학습 시작")
train_gnn(tr_data, val_data, te_data, tr_inds, val_inds, te_inds, args, data_config)

/usr/local/lib/python3.12/dist-packages/torch_geometric/loader/link_neighbor_loader.py:252: UserWarning: Using 'NeighborSampler' without a 'pyg-lib' installation is deprecated and will be removed soon. Please install 'pyg-lib' for accelerated neighborhood sampling
  neighbor_sampler = NeighborSampler(
100%|██████████| 620/620 [08:11<00:00,  1.26it/s]


## 8. TensorBoard (선택)

In [ ]:
%load_ext tensorboard
%tensorboard --logdir {BASELINE_DIR}/runs

## 8-1. runs 폴더 Drive에 저장 (선택)

런타임 초기화 시 runs 폴더가 사라지므로, 학습 후 Drive에 백업합니다.

In [ ]:
import shutil

runs_src = BASELINE_DIR / 'runs'
runs_dst = DRIVE_BASE / 'experiments' / 'multi_pna_eu' / 'runs'

if runs_src.exists():
    shutil.copytree(str(runs_src), str(runs_dst), dirs_exist_ok=True)
    print(f"저장 완료: {runs_dst}")
    print("저장된 run 목록:")
    for r in sorted(runs_dst.iterdir()):
        print(f"  {r.name}")
else:
    print("runs 폴더 없음. 학습을 먼저 실행하세요.")

## 9. 추론 (선택)

저장된 모델로 test set 추론만 할 경우 아래 셀을 실행하세요.
`args.unique_name`이 저장 시 이름과 일치해야 합니다.

In [ ]:
from inference import infer_gnn

args.inference = True
infer_gnn(tr_data, val_data, te_data, tr_inds, val_inds, te_inds, args, data_config)